In [ ]:
import json
from typing import List
import os

# Unstructured for documents parsing
from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title

# Langchain components
from langchain_core.documents import Document
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_ollama.llms import OllamaLLM
from langchain_core.prompts import ChatPromptTemplate
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from dotenv import load_dotenv

print("Successfully imported all libraries.")
load_dotenv()

Successfully imported all libraries.


False

In [7]:
def partition_document(file_path:str):
    """ Extract elements from a PDF using unstructured. """
    print(f"Partitioning document: {file_path}")

    elements = partition_pdf(
        filename = file_path,
        strategy = 'hi_res',
        infer_table_structure = True,
        extract_image_block_types = ["Image"],
        extract_image_block_to_payload = True
    )
    print(f"Extracted {len(elements)} elements from the document.")
    return elements

In [8]:
file_path = "docs\\NIPS-2017-attention-is-all-you-need-Paper.pdf"
elements = partition_document(file_path)

Partitioning document: docs\NIPS-2017-attention-is-all-you-need-Paper.pdf


No languages specified, defaulting to English.


preprocessor_config.json:   0%|          | 0.00/274 [00:00<?, ?B/s]

The `max_size` parameter is deprecated and will be removed in v4.26. Please specify in `size['longest_edge'] instead`.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/115M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

Extracted 183 elements from the document.


In [9]:
elements[31].to_dict()

{'type': 'NarrativeText',
 'element_id': 'f2d52981a12f76a79b4d07128795ef14',
 'text': 'Most competitive neural sequence transduction models have an encoder-decoder structure [5, 2, 29]. Here, the encoder maps an input sequence of symbol representations (x1,...,xn) to a sequence of continuous representations z = (z1,...,zn). Given z, the decoder then generates an output sequence (y1,...,ym) of symbols one element at a time. At each step the model is auto-regressive [9], consuming the previously generated symbols as additional input when generating the next.',
 'metadata': {'detection_class_prob': 0.9439999461174011,
  'is_extracted': 'true',
  'coordinates': {'points': ((np.float64(520.1356201171875),
     np.float64(2795.6119799999997)),
    (np.float64(520.1356201171875), np.float64(3056.16073)),
    (np.float64(2473.197998046875), np.float64(3056.16073)),
    (np.float64(2473.197998046875), np.float64(2795.6119799999997))),
   'system': 'PixelSpace',
   'layout_width': 2975,
   'layo

In [10]:
def create_chunks_by_titles(elements:List):
    """ Create intelligent chunks using title-based strategy."""
    print("Creating Smart Chunks...")
    chunks = chunk_by_title(elements, 
                            max_characters = 3000,
                            new_after_n_chars = 2400, 
                            combine_text_under_n_chars = 500)

    print("Created Smart Chunks")
    return chunks

In [11]:
chunks  = create_chunks_by_titles(elements)

Creating Smart Chunks...
Created Smart Chunks


In [12]:
chunks

In [13]:
len(chunks)

26

In [14]:
chunks[1].to_dict()

{'type': 'CompositeElement',
 'element_id': 'd351d9ed-af2d-4cdf-8763-bf5ad2ed4282',
 'text': '1 Introduction\n\nRecurrent neural networks, long short-term memory [12] and gated recurrent [7] neural networks in particular, have been ﬁrmly established as state of the art approaches in sequence modeling and transduction problems such as language modeling and machine translation [29, 2, 5]. Numerous efforts have since continued to push the boundaries of recurrent language models and encoder-decoder architectures [31, 21, 13].\n\n∗Equal contribution. Listing order is random. Jakob proposed replacing RNNs with self-attention and started\n\nthe effort to evaluate this idea. Ashish, with Illia, designed and implemented the ﬁrst Transformer models and has been crucially involved in every aspect of this work. Noam proposed scaled dot-product attention, multi-head attention and the parameter-free position representation and became the other person involved in nearly every detail. Niki designed, i

In [15]:
chunks[4].metadata.orig_elements

In [16]:
chunks[11].metadata.orig_elements[0].to_dict()

{'type': 'Table',
 'element_id': '25b9167c201f0fe4485e4890224185e5',
 'text': 'Layer Type Complexity per Layer Sequential Maximum Path Length Operations Self-Attention O(n2 · d) O(1) O(1) Recurrent O(n · d2) O(n) O(n) Convolutional O(k · n · d2) O(1) O(logk(n)) Self-Attention (restricted) O(r · n · d) O(1) O(n/r)',
 'metadata': {'detection_class_prob': 0.9264122247695923,
  'is_extracted': 'true',
  'coordinates': {'points': ((np.float64(565.9484252929688),
     np.float64(553.2286987304688)),
    (np.float64(565.9484252929688), np.float64(907.3110961914062)),
    (np.float64(2380.669921875), np.float64(907.3110961914062)),
    (np.float64(2380.669921875), np.float64(553.2286987304688))),
   'system': 'PixelSpace',
   'layout_width': 2975,
   'layout_height': 3850},
  'last_modified': '2026-05-23T11:50:47',
  'text_as_html': '<table><thead><tr><th>Layer Type</th><th>Complexity per Layer</th><th>Sequential Operations</th><th>Maximum Path Length</th></tr></thead><tbody><tr><td>Self-Atten

In [17]:
def separate_content_types(chunk):
    """ Separate text and image content from a chunk. """
    content_data = {
        "text": chunk.text,
        "tables": [],
        "images": [],
        "types": ['text']
    }
    
    if hasattr(chunk, "metadata") and hasattr(chunk.metadata, "orig_elements"):     
        for element in chunk.metadata.orig_elements:
            element_type = type(element).__name__

            # Handle Tables
            if element_type == "Table":
                content_data["types"].append("table")
                table_html = getattr(element.metadata, 'text_as_html', element.text)
                content_data["tables"].append(table_html)

            # Handle Images
            elif element_type == "Image":
                if hasattr(element, 'metadata') and hasattr(element.metadata, 'image_base64'):
                    content_data["types"].append("image")                    
                    content_data["images"].append(element.metadata.image_base64)
        
        content_data['types'] = list(set(content_data['types']))
        return content_data

In [18]:
llm = OllamaLLM(model="multimodal-vision", temperature=0.7)

In [19]:
def create_AI_enhanced_summary(text:str, tables:List[str], images:List[str]) -> str:
    """ Create an AI-enhanced summary using Ollama LLM. """
    try:
        # Initialize Ollama LLM (Need vision model for image)
        
        llm = ChatOllama(
            model="gemma3:4b",
            temperature=0,
            # num_ctx=2048,
            # num_predict=500,
        )

        # Build the prompt with text and tables
        prompt_text = f""" You are creating a searchable description for document content retrieval.

        CONTENT TO ANALYZE:
        TEXT CONTENT:
        {text} 
        """
        
        # Add tables if present
        if tables:
            for i, table in enumerate(tables):
                prompt_text += f"\nTABLE {i+1}:\n{table}\n"
        
        # Expanding the prompt's instructions
        prompt_text += """
            YOUR TASK:
            Generate a comprehensive, searchable description that covers:

            1. Key facts, numbers, and data points from text and tables
            2. Main topics and concepts discussed  
            3. Questions this content could answer
            4. Visual content analysis (charts, diagrams, patterns in images)
            5. Alternative search terms users might use

            Make it detailed and searchable - prioritize findability over brevity.

            SEARCHABLE DESCRIPTION:
            """
                
        # Build message content starting with text
        message_content = [
            {
                'type': "text",
                'text': prompt_text
            }
        ]


        # Add images
        for image_base_64 in images:
            message_content.append(
                {
                'type': "image_url",
                'image_url': {'url': f"data:image/jpeg;base64,{image_base_64}"}
                }
                )

        # Send to LLM and get response
        message = HumanMessage(content=message_content)
        response = llm.invoke([message])      

        return response.content
    
    except Exception as e:
        print(f"     ❌ AI summary failed: {e}")
        # Fallback to simple summary
        summary = f"{text[:300]}..."
        if tables:
            summary += f" [Contains {len(tables)} table(s)]"
        if images:
            summary += f" [Contains {len(images)} image(s)]"
        return summary

In [20]:
def summarise_chunks(chunks):
    """ Summarize a chunk with AI summaries. """

    langchain_documents = []
    total_chunks = len(chunks)

    for i, chunk in enumerate(chunks):
        print(f"Summarizing chunk {i+1}/{total_chunks}...")

        # Separating chunk's content types
        content_data = separate_content_types(chunk)

        # Debugging prints
        print(f"     Types found: {content_data['types']}")
        print(f"     Tables: {len(content_data['tables'])}, Images: {len(content_data['images'])}")
        
        # Creating AI-enhanced summary for the chunk (content_data)
        if content_data['tables'] or content_data['images']:
            print(f"     → Creating AI summary for mixed content...")
            try:
                enhanced_content = create_AI_enhanced_summary(content_data['text'], content_data['tables'], content_data['images'])
                print(f"     → AI summary created successfully.")
                print(f"     Summary preview: {enhanced_content[:200]}...")
            except Exception as e:
                print(f"     ❌ AI summary failed: {e}")
                enhanced_content = content_data['text']
        else:
            print(f"     → Using raw text (no tables/images)")
            enhanced_content = content_data['text']

        # Create a Langchain Document with the summary and metadata
        doc = Document(
            page_content=enhanced_content,
            metadata={
                "original_content": json.dumps({
                    "tables": content_data['tables'],
                    "images": content_data['images'],
                    "types": content_data['types']
                })
            }
        )
        langchain_documents.append(doc)

    print(f"✅ Processed {len(langchain_documents)} chunks")
    return langchain_documents

# Testing Functions

In [21]:
chunks[5].to_dict()

{'type': 'CompositeElement',
 'element_id': '39bb2834-0e76-4a0d-b2ee-7a5ab16dafd1',
 'text': '3.2 Attention\n\nAn attention function can be described as mapping a query and a set of key-value pairs to an output, where the query, keys, values, and output are all vectors. The output is computed as a weighted sum of the values, where the weight assigned to each value is computed by a compatibility function of the query with the corresponding key.\n\n3.2.1 Scaled Dot-Product Attention\n\nWe call our particular attention "Scaled Dot-Product Attention" (Figure 2). The input consists of queries and keys of dimension dk, and values of dimension dv. We compute the dot products of the\n\n3\n\nScaled Dot-Product Attention\n\nMulti-Head Attention\n\nEEE Scaled Dot-Product | Attention 4\n\nFigure 2: (left) Scaled Dot-Product Attention. (right) Multi-Head Attention consists of several attention layers running in parallel.\n\n√\n\nquery with all keys, divide each by dk, and apply a softmax function t

In [22]:
chunks[4].metadata.orig_elements

In [23]:
separated_content = separate_content_types(chunks[4])
separated_content

{'text': '3.1 Encoder and Decoder Stacks\n\nEncoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two sub-layers. The ﬁrst is a multi-head self-attention mechanism, and the second is a simple, position-\n\n2\n\nOutput Probabilities Add & Norm Feed Forward Add & Norm Multi-Head Attention a, Add & Norm Nx Add & Norm Feed Forward Nx | -Casda Nom] Add & Norm VWEeea Multi-Head Multi-Head Attention Attention Sy ae, Se a, Positional @ Encoding @ Positional @ q Encoding Input Embedding Inputs Outputs (shifted right) Output Embedding\n\nFigure 1: The Transformer - model architecture.\n\nwise fully connected feed-forward network. We employ a residual connection [10] around each of the two sub-layers, followed by layer normalization [1]. That is, the output of each sub-layer is LayerNorm(x + Sublayer(x)), where Sublayer(x) is the function implemented by the sub-layer itself. To facilitate these residual connections, all sub-layers in the model, as well as the embed

In [24]:
chunks[11].metadata.orig_elements[0].to_dict()

{'type': 'Table',
 'element_id': '25b9167c201f0fe4485e4890224185e5',
 'text': 'Layer Type Complexity per Layer Sequential Maximum Path Length Operations Self-Attention O(n2 · d) O(1) O(1) Recurrent O(n · d2) O(n) O(n) Convolutional O(k · n · d2) O(1) O(logk(n)) Self-Attention (restricted) O(r · n · d) O(1) O(n/r)',
 'metadata': {'detection_class_prob': 0.9264122247695923,
  'is_extracted': 'true',
  'coordinates': {'points': ((np.float64(565.9484252929688),
     np.float64(553.2286987304688)),
    (np.float64(565.9484252929688), np.float64(907.3110961914062)),
    (np.float64(2380.669921875), np.float64(907.3110961914062)),
    (np.float64(2380.669921875), np.float64(553.2286987304688))),
   'system': 'PixelSpace',
   'layout_width': 2975,
   'layout_height': 3850},
  'last_modified': '2026-05-23T11:50:47',
  'text_as_html': '<table><thead><tr><th>Layer Type</th><th>Complexity per Layer</th><th>Sequential Operations</th><th>Maximum Path Length</th></tr></thead><tbody><tr><td>Self-Atten

In [25]:
el = chunks[11].metadata.orig_elements[0]
res = getattr(el.metadata, 'text_as_html', el.text) or el.text
print(res)

<table><thead><tr><th>Layer Type</th><th>Complexity per Layer</th><th>Sequential Operations</th><th>Maximum Path Length</th></tr></thead><tbody><tr><td>Self-Attention</td><td>O(n? - d)</td><td>O(1)</td><td>O(1)</td></tr><tr><td>Recurrent</td><td>O(n - d?)</td><td>O(n)</td><td>O(n)</td></tr><tr><td>Convolutional</td><td>O(k-n-d?)</td><td>O(1)</td><td>O(log, (n))</td></tr><tr><td>Self-Attention (restricted)</td><td>O(r-n-d)</td><td>O(1)</td><td>O(n/r)</td></tr></tbody></table>


In [26]:
separated_content = separate_content_types(chunks[11])
separated_content

{'text': 'Layer Type Complexity per Layer Sequential Maximum Path Length Operations Self-Attention O(n2 · d) O(1) O(1) Recurrent O(n · d2) O(n) O(n) Convolutional O(k · n · d2) O(1) O(logk(n)) Self-Attention (restricted) O(r · n · d) O(1) O(n/r)',
 'tables': ['<table><thead><tr><th>Layer Type</th><th>Complexity per Layer</th><th>Sequential Operations</th><th>Maximum Path Length</th></tr></thead><tbody><tr><td>Self-Attention</td><td>O(n? - d)</td><td>O(1)</td><td>O(1)</td></tr><tr><td>Recurrent</td><td>O(n - d?)</td><td>O(n)</td><td>O(n)</td></tr><tr><td>Convolutional</td><td>O(k-n-d?)</td><td>O(1)</td><td>O(log, (n))</td></tr><tr><td>Self-Attention (restricted)</td><td>O(r-n-d)</td><td>O(1)</td><td>O(n/r)</td></tr></tbody></table>'],
 'images': [],
 'types': ['table', 'text']}

In [27]:
create_AI_enhanced_summary(separated_content['text'], separated_content['tables'], separated_content['images'])

'## Document Description: Neural Network Layer Complexity Analysis – A Comparative Study\n\nThis document presents a detailed analysis of the computational complexity of various neural network layer types, focusing on their performance characteristics and operational requirements. The core of the analysis is presented in Table 1, alongside supplementary textual descriptions, providing a comprehensive understanding of the trade-offs involved in utilizing different layer architectures within deep learning models.\n\n**1. Key Facts, Numbers, and Data Points:**\n\n* **Layer Types Analyzed:** The document examines four primary layer types: Self-Attention, Recurrent, Convolutional, and Restricted Self-Attention.\n* **Complexity Metrics:**  The analysis utilizes Big O notation to quantify the computational complexity of each layer type, expressed as a function of:\n    * **n:** Input sequence length\n    * **d:** Dimensionality of the input data\n    * **k:** Number of kernels (convolutional 

In [28]:
import base64


def image_to_base64(image_path: str) -> str:
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")


image_base64 = image_to_base64("docs/test.png")

llm = ChatOllama(
    model="gemma3:4b",
    temperature=0,
    num_ctx=2048,
    num_predict=300,
)

message = HumanMessage(
    content=[
        {
            "type": "text",
            "text": "Tell me about this image.",
        },
        {
            "type": "image_url",
            "image_url": {
                "url": f"data:image/png;base64,{image_base64}"
            },
        },
    ]
)

response = llm.invoke([message])

print(response.content)

Okay, let's break down this image, which illustrates the architecture of a Transformer model. Here's a description of the components and how they're connected:

**Overall Structure:**

The image represents a simplified version of the core architecture of a Transformer, a neural network architecture that has revolutionized natural language processing (NLP) and is now being applied to other domains like computer vision.

**Components and Flow:**

1. **Inputs:** The process starts with a set of "Inputs" (labeled as N x). These are the initial data that the model is processing.

2. **Input Embedding:** The inputs are first converted into a numerical representation through "Input Embedding." This is a standard technique in neural networks where each input element (e.g., a word) is mapped to a vector of numbers.

3. **Positional Encoding:**  Crucially, Transformers don't inherently understand the order of the input sequence.  "Positional Encoding" is added to the input embeddings. This provi

In [29]:
# proessing chunks to create AI-enhanced summaries
processed_chunks = summarise_chunks(chunks)
processed_chunks

Summarizing chunk 1/26...
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
Summarizing chunk 2/26...
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
Summarizing chunk 3/26...
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
Summarizing chunk 4/26...
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
Summarizing chunk 5/26...
     Types found: ['image', 'text']
     Tables: 0, Images: 1
     → Creating AI summary for mixed content...
     → AI summary created successfully.
     Summary preview: Here's a detailed, searchable description of the provided text content, designed for effective information retrieval:

**SEARCHABLE DESCRIPTION: Transformer Model Architecture - Encoder and Decoder**
...
Summarizing chunk 6/26...
     Types found: ['image', 'text']
     Tables: 0, Images: 2
     → Creating AI summary for

[Document(metadata={'original_content': '{"tables": [], "images": [], "types": ["text"]}'}, page_content='Attention Is All You Need\n\nAshish Vaswani∗ Google Brain avaswani@google.com\n\nNoam Shazeer∗ Google Brain noam@google.com\n\nNiki Parmar∗\n\nGoogle Research nikip@google.com\n\nJakob Uszkoreit∗ Google Research usz@google.com\n\nLlion Jones∗\n\nGoogle Research llion@google.com\n\nAidan N. Gomez∗ † University of Toronto aidan@cs.toronto.edu\n\nŁukasz Kaiser∗\n\nGoogle Brain lukaszkaiser@google.com\n\nIllia Polosukhin∗ ‡\n\nillia.polosukhin@gmail.com\n\nAbstract\n\nThe dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments on two machine tr

In [31]:
processed_chunks[0]

Document(metadata={'original_content': '{"tables": [], "images": [], "types": ["text"]}'}, page_content='Attention Is All You Need\n\nAshish Vaswani∗ Google Brain avaswani@google.com\n\nNoam Shazeer∗ Google Brain noam@google.com\n\nNiki Parmar∗\n\nGoogle Research nikip@google.com\n\nJakob Uszkoreit∗ Google Research usz@google.com\n\nLlion Jones∗\n\nGoogle Research llion@google.com\n\nAidan N. Gomez∗ † University of Toronto aidan@cs.toronto.edu\n\nŁukasz Kaiser∗\n\nGoogle Brain lukaszkaiser@google.com\n\nIllia Polosukhin∗ ‡\n\nillia.polosukhin@gmail.com\n\nAbstract\n\nThe dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments on two machine tra

In [35]:
def export_chunks_to_json(chunks, filename="results/processed_chunks.json"):
    """ Export processed chunks to a JSON file. """

    # Create results/ directory if it doesn't exist
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    export_data = []
    for chunk in chunks:
        chunk_data = ({
            "summary": chunk.page_content,
            "metadata": {
                "original_content": json.loads(chunk.metadata.get("original_content", "{}"))
            }
        })
        export_data.append(chunk_data)
    
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(export_data, f, ensure_ascii=False, indent=2)
    
    print(f"✅ Exported {len(chunks)} chunks to {filename}")
    return export_data

In [36]:
json_data = export_chunks_to_json(processed_chunks)

✅ Exported 26 chunks to results/processed_chunks.json


In [44]:
def create_vector_store(documents, collection_name= "multimodal_chunks", persist_directory="dbv1/chroma_db"):
    """ Create a Chroma vector store from processed chunks. """
    print("🔮 Creating embeddings and storing in ChromaDB...")

    # Initialize Ollama Embeddings (using a vision-capable model for multimodal content)
    embedding_model = OllamaEmbeddings(model="bge-m3")

    # Create Chroma vector store
    vector_store = Chroma.from_documents(
        documents=documents,
        collection_name = collection_name,
        embedding = embedding_model,
        persist_directory= persist_directory,
        collection_metadata = {"hnsw:space" : "cosine",}
    )
    print("--- Finished creating vector store ---")
    
    print(f"✅ Vector store created and saved to {persist_directory}")
    return vector_store

In [45]:
vector_store = create_vector_store(processed_chunks)

🔮 Creating embeddings and storing in ChromaDB...
--- Finished creating vector store ---
✅ Vector store created and saved to dbv1/chroma_db


# TESTS


In [49]:
# After your retrieval
query = "What are the two main components of the Transformer architecture? "
retriever = vector_store.as_retriever(search_kwargs={"k": 3})
chunks = retriever.invoke(query)

# Export to JSON
export_chunks_to_json(chunks, "results/rag_results.json")

✅ Exported 3 chunks to results/rag_results.json


[{'summary': 'In this work we propose the Transformer, a model architecture eschewing recurrence and instead relying entirely on an attention mechanism to draw global dependencies between input and output. The Transformer allows for signiﬁcantly more parallelization and can reach a new state of the art in translation quality after being trained for as little as twelve hours on eight P100 GPUs.\n\n2 Background\n\nThe goal of reducing sequential computation also forms the foundation of the Extended Neural GPU [20], ByteNet [15] and ConvS2S [8], all of which use convolutional neural networks as basic building block, computing hidden representations in parallel for all input and output positions. In these models, the number of operations required to relate signals from two arbitrary input or output positions grows in the distance between positions, linearly for ConvS2S and logarithmically for ByteNet. This makes it more difﬁcult to learn dependencies between distant positions [11]. In the 

In [ ]:
def run_complete_ingestion_pipeline(file_path:str):
    """Run the complete RAG ingestion pipeline"""
    print("🚀 Starting RAG Ingestion Pipeline")
    print("=" * 50)

    #Step1: Partition document
    res_elements = partition_document(file_path)

    #Step2: Chunks
    res_chunks  = create_chunks_by_titles(res_elements)

    #Step3: AI-enhanced summarization
    res_processed_chunks = summarise_chunks(res_chunks)

    #Step4: Vector store 
    db = create_vector_store(res_processed_chunks)

    print("🎉 Pipeline completed successfully!")
    return db

In [ ]:
db = run_complete_ingestion_pipeline(file_path)

In [47]:
# Query the vector store
query = "How many attention heads does the Transformer use, and what is the dimension of each head? "

retriever = vector_store.as_retriever(search_kwargs={"k": 3})
chunks = retriever.invoke(query)

# Final RUN

In [50]:
def generating_final_answer(query, relevant_chunks):
    """Generate final answer using multimodal content"""
    try:

        # Initializing LLM (Needs vision model for images)
        
        llm = ChatOllama(
            model="gemma3:4b",
            temperature=0,
            # num_ctx=2048,
            # num_predict=500,
        )

        prompt_text = f"""Based on the following documents, please answer this question: {query}

        CONTENT TO ANALYZE:
        """

        for i, chunk in enumerate(relevant_chunks):
            prompt_text += f"\nDOCUMENT {i+1}:\n"
            if 'original_content' in chunk.metadata:
                original_data = json.loads(chunk.metadata['original_content'])

                # Add Raw text
                if 'text' in original_data:
                    prompt_text += f"\nTEXT:\n{original_data['text']}\n"
                
                # Add tables
                tables_html = original_data.get("tables_html", [])
                if tables_html:
                    prompt_text += "TABLES:\n"
                    for j, table in enumerate(tables_html):
                        prompt_text += f"Table {j+1}:\n{table}\n\n"
            prompt_text += '\n'
        
        prompt_text += """
    Please provide a clear, comprehensive answer using the text, tables, and images above. If the documents don't contain sufficient information to answer the question, say "I don't have enough information to answer that question based on the provided documents."

    ANSWER:"""

        # Build message content starting with text
        message_content = [{"type": "text", "text": prompt_text}]

        # Add All images from all chunks
        for chunk in relevant_chunks:
            if 'original_content' in chunk.metadata:
                original_data = json.loads(chunk.metadata['original_content'])
                images = original_data.get("images", [])
                for image_base_64 in images:
                    message_content.append(
                        {
                        'type': "image_url",
                        'image_url': {'url': f"data:image/jpeg;base64,{image_base_64}"}
                        }
                    )
        # Send to AI and get response
        message = HumanMessage(content=message_content)
        response = llm.invoke([message])

        return response.content
    
    except Exception as e:
        print(f"❌ Answer generation failed: {e}")
        return "Sorry, I encountered an error while generating the answer."
                
            



In [52]:
query = "In Recurrent layer types, what are the respective complexity per layer, and maximum path length"
print(f"User> {query}")
print(f"AI> {generating_final_answer(query, chunks)}")


User> In Recurrent layer types, what are the respective complexity per layer, and maximum path length
AI> Okay, let's analyze the provided documents to determine the complexity per layer and maximum path length of recurrent layer types.

**Analysis:**

The documents describe different recurrent layer types and their associated complexities. Here's a breakdown based on the information presented:

*   **Simple RNN:** The document states that the complexity of a simple RNN is O(N), where N is the sequence length. The maximum path length is N.

*   **LSTM:** The document indicates that the complexity of an LSTM is O(N^2), where N is the sequence length. The maximum path length is N.

*   **GRU:** The document states that the complexity of a GRU is O(N^2), where N is the sequence length. The maximum path length is N.

**Summary Table:**

| Recurrent Layer Type | Complexity per Layer | Maximum Path Length |
| -------------------- | --------------------- | -------------------- |
| Simple RNN 

In [53]:
query = "How many layers does the base transformer model use in both encoder and decoder"
print(f"User> {query}")
print(f"AI> {generating_final_answer(query, chunks)}")


User> How many layers does the base transformer model use in both encoder and decoder
AI> Here's an analysis of the transformer model layers based on the provided documents:

**Answer:**

The base transformer model uses 6 layers in both the encoder and the decoder.

**Explanation:**

*   **Document 1** describes the overall architecture of the transformer, stating that it consists of an encoder and a decoder, each with 6 layers.
*   **Document 2** further elaborates on the encoder, explicitly stating "The encoder consists of 6 layers."
*   **Document 3** similarly states "The decoder consists of 6 layers."

Therefore, the model employs 6 layers for both the encoder and the decoder.
